# arXiv Computer Science 최근 6개월 논문 수집

arXiv API에서 Computer Science(`cs.*`) 카테고리에 해당하고, 노트북 실행일 기준 최근 6개월 이내에 제출된 논문을 조회해 JSONL 파일로 저장합니다.

> arXiv API 이용 시 요청 사이에 3초 이상 간격을 두며, 기본 수집 상한은 1,000건입니다. `MAX_RESULTS`를 조정해 수집 규모를 바꿀 수 있습니다.

In [4]:
from calendar import monthrange
from datetime import date, datetime, timezone
from pathlib import Path
import json
import ssl
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

MAX_RESULTS = 1000
PAGE_SIZE = 50
REQUEST_INTERVAL_SECONDS = 5.0
MAX_RETRIES = 6
BACKOFF_BASE_SECONDS = 5.0
BACKOFF_MAX_SECONDS = 120.0
API_URL = 'https://export.arxiv.org/api/query'

def subtract_months(day, months):
    """월말에서도 안전하게 지정한 개월 수를 뺍니다."""
    month_index = day.year * 12 + day.month - 1 - months
    year, month_zero_based = divmod(month_index, 12)
    month = month_zero_based + 1
    return date(year, month, min(day.day, monthrange(year, month)[1]))

END_DATE = date.today()
START_DATE = subtract_months(END_DATE, 6)
DATE_QUERY = f'submittedDate:[{START_DATE:%Y%m%d}0000 TO {END_DATE:%Y%m%d}2359]'
SEARCH_QUERY = f'cat:cs.* AND {DATE_QUERY}'

# 노트북을 어디서 실행해도 프로젝트 루트의 data 폴더에 저장
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'arxiv_cs_recent_6months.jsonl'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f'조회 기간: {START_DATE} ~ {END_DATE}')
print(f'검색식: {SEARCH_QUERY}')
print(f'저장 위치: {OUTPUT_PATH.resolve()}')

조회 기간: 2026-03-12 ~ 2026-09-12
검색식: cat:cs.* AND submittedDate:[202603120000 TO 202609122359]
저장 위치: C:\Users\oh\Desktop\arxiv_graph_RAG\data\arxiv_cs_recent_6months.jsonl


In [5]:
ATOM_NS = {'atom': 'http://www.w3.org/2005/Atom'}
ARXIV_NS = {'arxiv': 'http://arxiv.org/schemas/atom'}

def text_or_none(parent, path, namespaces=ATOM_NS):
    node = parent.find(path, namespaces)
    return node.text.strip() if node is not None and node.text else None

def parse_entry(entry):
    authors = []
    for author in entry.findall('atom:author', ATOM_NS):
        name = text_or_none(author, 'atom:name')
        if name:
            authors.append(name)
    categories = [
        node.attrib['term']
        for node in entry.findall('atom:category', ATOM_NS)
        if 'term' in node.attrib
    ]
    links = {
        link.attrib.get('rel', 'alternate'): link.attrib.get('href')
        for link in entry.findall('atom:link', ATOM_NS)
    }
    published = text_or_none(entry, 'atom:published')
    primary_category_node = entry.find('arxiv:primary_category', ARXIV_NS)
    primary_category = primary_category_node.attrib.get('term') if primary_category_node is not None else None
    return {
        'id': text_or_none(entry, 'atom:id'),
        'title': ' '.join((text_or_none(entry, 'atom:title') or '').split()),
        'abstract': ' '.join((text_or_none(entry, 'atom:summary') or '').split()),
        'authors': authors,
        'categories': categories,
        'primary_category': primary_category,
        'published': published,
        'updated': text_or_none(entry, 'atom:updated'),
        'doi': text_or_none(entry, 'arxiv:doi', ARXIV_NS),
        'pdf_url': links.get('related') or links.get('alternate'),
        'source': 'arxiv',
        'collection_window': {
            'start': START_DATE.isoformat(),
            'end': END_DATE.isoformat(),
        },
    }

def _retry_delay(error, attempt):
    retry_after = error.headers.get('Retry-After') if getattr(error, 'headers', None) else None
    if retry_after:
        try:
            return max(float(retry_after), REQUEST_INTERVAL_SECONDS)
        except ValueError:
            pass
    return min(BACKOFF_BASE_SECONDS * (2 ** attempt), BACKOFF_MAX_SECONDS)

_last_request_at = 0.0

def _urlopen_with_retry(request, timeout):
    global _last_request_at
    for attempt in range(MAX_RETRIES + 1):
        elapsed = time.monotonic() - _last_request_at
        if elapsed < REQUEST_INTERVAL_SECONDS:
            time.sleep(REQUEST_INTERVAL_SECONDS - elapsed)
        try:
            _last_request_at = time.monotonic()
            return urllib.request.urlopen(request, timeout=timeout, context=ssl.create_default_context())
        except urllib.error.HTTPError as error:
            if error.code not in {429, 500, 502, 503, 504} or attempt >= MAX_RETRIES:
                raise
            delay = _retry_delay(error, attempt)
            print(f'HTTP {error.code}: {delay:.0f}珥????ъ떆??({attempt + 1}/{MAX_RETRIES})')
            error.close()
            time.sleep(delay)

def fetch_page(start=0, max_results=PAGE_SIZE):
    params = {
        'search_query': SEARCH_QUERY,
        'start': start,
        'max_results': max_results,
        'sortBy': 'submittedDate',
        'sortOrder': 'descending',
    }
    url = f'{API_URL}?{urllib.parse.urlencode(params)}'
    request = urllib.request.Request(url, headers={'User-Agent': 'arxiv-cs-jsonl/1.0'})
    with _urlopen_with_retry(request, timeout=60) as response:
        root = ET.fromstring(response.read())
    return [parse_entry(entry) for entry in root.findall('atom:entry', ATOM_NS)]

In [6]:
papers = []
seen_ids = set()
for start in range(0, MAX_RESULTS, PAGE_SIZE):
    page = fetch_page(start=start, max_results=min(PAGE_SIZE, MAX_RESULTS - start))
    for paper in page:
        if paper['id'] and paper['id'] not in seen_ids:
            seen_ids.add(paper['id'])
            papers.append(paper)
    print(f'{len(papers)}개 수집 완료')
    with OUTPUT_PATH.open('w', encoding='utf-8') as file:
        for paper in papers:
            file.write(json.dumps(paper, ensure_ascii=False) + '\n')
    if len(papers) >= MAX_RESULTS or len(page) < PAGE_SIZE:
        break


papers = papers[:MAX_RESULTS]
with OUTPUT_PATH.open('w', encoding='utf-8') as file:
    for paper in papers:
        file.write(json.dumps(paper, ensure_ascii=False) + '\n')

print(f'완료: {len(papers)}개 논문을 {OUTPUT_PATH.resolve()}에 저장했습니다.')

HTTPError: HTTP Error 429: Too Many Requests

In [ ]:
# JSONL 저장 결과 및 기간 조건 검증
with OUTPUT_PATH.open(encoding='utf-8') as file:
    saved_papers = [json.loads(line) for line in file if line.strip()]

assert len(saved_papers) == len(papers)
assert len({paper['id'] for paper in saved_papers}) == len(saved_papers)
assert all(paper['id'] and paper['title'] and paper['abstract'] for paper in saved_papers)
assert all(START_DATE.isoformat() <= paper['published'][:10] <= END_DATE.isoformat() for paper in saved_papers)
print(f'검증 완료: {len(saved_papers)}개 레코드, 중복 없음, 최근 6개월 범위 확인')
display(saved_papers[:3])